# 11 — Model comparison, judged only on tests batch can't inflate

Does a more flexible or more structured model find more real aging signal than the linear baseline? In this atlas a model can score well on unseen mice by reading the processing batch, so models are **never ranked by ordinary held-out AUROC**. Two tests only:

| Test | Question | Why batch can't help |
| --- | --- | --- |
| **cross-age** | trained on some age groups, scored on unseen ones (up to 6 splits per row, the same for every model) | each age group is its own batch |
| **transfer** | trained on all TMS mice, scored on Kimmel et al. 2019 (another lab) | the model never saw that lab's batch |

Rows: the 19 tissue × cell types matched to Kimmel in 10. Every model setting is fixed in `src/models.py` before anything is run; nothing is tuned on these tests. Part 2 compares clocks the same way.

| Model | Kind | Idea |
| --- | --- | --- |
| logistic (baseline) | linear, per cell | 05a, L2, C = 0.1 |
| elastic net | linear, sparse | L1 + L2: a short gene panel (~40 genes) might generalize better |
| LightGBM | gradient-boosted trees | can learn non-linear rules |
| mouse-level | linear, one sample per mouse | trains at the true sample size |
| attention MIL | neural network (PyTorch) | a mouse is a bag of cells with one label; attention learns which cells carry it |

### 1. Setup

In [ ]:
import sys, time
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut
from src import aging as ag
from src import models as M

A = ag.paths("atlas")
atlas = pd.read_csv(A.figure("atlas.csv"))
atlas["robust"] = atlas["verdict"].eq("age signal") & ~atlas["tech_confounded"].astype(bool)
rep = pd.read_csv(A.figure("replication_kimmel.csv"))      # the 19 matched rows from 10
KIM = ag.ROOT / "data" / "external" / "Kimmel_GR_2019_data"

### 2. Kimmel, normalized as in 10

In [ ]:
kim = {}
for t in ["lung", "kidney", "spleen"]:
    k = sc.read_h5ad(KIM / f"{t}.h5ad")
    sc.pp.normalize_total(k, target_sum=1e4)
    sc.pp.log1p(k)
    k.obs["mouse"] = k.obs["age"].astype(str) + "-" + k.obs["animal"].astype(str)
    kim[t] = k

### 3. Score every model on every row

Per row: genes = TMS variable genes minus ambient, technical and sex genes, restricted to genes Kimmel also has. Each score is reported per cell and per mouse (mean cell score per mouse, then AUROC over mice). About 20 minutes, mostly the neural network.

In [ ]:
def mouse_auc(y, s, mice):
    d = pd.DataFrame({"y": y, "s": s, "m": mice}).groupby("m").mean()
    return roc_auc_score(d["y"], d["s"]) if d["y"].nunique() == 2 else np.nan

rng = np.random.default_rng(0)
rows, loaded = [], {}
for r in rep.itertuples():
    if r.tissue not in loaded:
        loaded = {r.tissue: sc.read_h5ad(ag.paths(r.tissue).processed)}
    a, k = loaded[r.tissue], kim[r.kim_tissue]
    feat = ag.feature_mask(a, "bio") & a.var_names.isin(k.var_names)
    genes = a.var_names[feat]
    X, y, mice, ages = ag.prep(a, r.cell_type, features=feat, return_ages=True, **ag.main_kw(r.cell_type, r.tissue))
    splits = [s for s in ag.cross_age_splits(set(ages))
              if len(set(y[np.isin(ages, list(s[0]))])) == 2 and len(set(y[np.isin(ages, list(s[1]))])) == 2]
    if len(splits) > 6:
        splits = [splits[i] for i in sorted(rng.choice(len(splits), 6, replace=False))]
    kk = k[(k.obs[r.kim_col] == r.kim_label).to_numpy()]
    Xk = kk[:, genes].X
    Xk = Xk.toarray() if hasattr(Xk, "toarray") else np.asarray(Xk)
    yk = kk.obs["age"].eq("old").to_numpy().astype(int)
    mk = kk.obs["mouse"].to_numpy()
    for name, make in M.CLASSIFIERS.items():
        t0 = time.time()
        cc, cm = [], []
        for tr_ages, te_ages in splits:
            tr, te = np.isin(ages, list(tr_ages)), np.isin(ages, list(te_ages))
            s = make().fit(X[tr], y[tr], mice[tr]).predict(X[te])
            cc.append(roc_auc_score(y[te], s)); cm.append(mouse_auc(y[te], s, mice[te]))
        sk = make().fit(X, y, mice).predict(Xk)
        rows.append({"tissue": r.tissue, "cell_type": r.cell_type, "robust": r.robust, "model": name,
                     "splits": len(splits), "cross_cell": np.mean(cc), "cross_mouse": np.nanmean(cm),
                     "transfer_cell": roc_auc_score(yk, sk), "transfer_mouse": mouse_auc(yk, sk, mk),
                     "seconds": time.time() - t0})
    print(f"{r.tissue}: {r.cell_type} ({len(splits)} splits)")
scores = pd.DataFrame(rows)
scores.round(4).to_csv(A.figure("models_classifiers.csv"), index=False)

### 4. Which model generalizes?

Mean over the 19 rows, and a paired test against the baseline on each row (Wilcoxon signed-rank; `wins` = rows where the model beats the baseline). A model only counts as better if it wins on **both** tests.

In [ ]:
base = scores[scores["model"] == "logistic (baseline)"].set_index(["tissue", "cell_type"])
summary = []
for name, d in scores.groupby("model", sort=False):
    d = d.set_index(["tissue", "cell_type"])
    out = {"model": name, "seconds_per_row": d["seconds"].mean()}
    for m in ["cross_cell", "cross_mouse", "transfer_cell", "transfer_mouse"]:
        out[m] = d[m].mean()
        if name != "logistic (baseline)":
            diff = (d[m] - base.loc[d.index, m]).dropna()
            out[f"{m}_wins"] = int((diff > 0).sum())
            out[f"{m}_p"] = wilcoxon(diff).pvalue if (diff != 0).any() else np.nan
    summary.append(out)
summary = pd.DataFrame(summary).set_index("model")
summary.round(3).to_csv(A.figure("models_classifiers_summary.csv"))
summary[["cross_cell", "cross_cell_wins", "cross_cell_p", "transfer_cell", "transfer_cell_wins", "transfer_cell_p",
         "cross_mouse", "transfer_mouse", "seconds_per_row"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
order = list(M.CLASSIFIERS)
for ax, m, title in [(axes[0], "cross_cell", "unseen age groups (TMS)"), (axes[1], "transfer_cell", "another lab (Kimmel)")]:
    sns.boxplot(data=scores, x=m, y="model", order=order, ax=ax, color="lightgray", showfliers=False)
    sns.stripplot(data=scores, x=m, y="model", order=order, hue="robust", ax=ax, size=5, jitter=0.15)
    ax.axvline(0.5, color="gray", lw=0.6); ax.set_title(title); ax.set_xlabel("AUROC (cells)")
fig.savefig(A.figure("models_classifiers.png"), dpi=200, bbox_inches="tight")

### 5. Does the inside-TMS score predict transfer, for any model?

If a model's cross-age score were a good guide, rows (and models) that score high inside TMS would also transfer best.

In [ ]:
pd.DataFrame({name: {"spearman_cross_vs_transfer": spearmanr(d["cross_cell"], d["transfer_cell"])[0],
                     "p": spearmanr(d["cross_cell"], d["transfer_cell"])[1]}
              for name, d in scores.groupby("model", sort=False)}).T.round(3)

### 6. Clocks: does a different model fix the 21-month inversion?

Every row with mouse-held-out AUROC ≥ 0.8 in every tissue (as 05c), adult mice, leave-one-mouse-out. Besides error in months, the test that matters: hold out each age group that has an adult age on both sides and ask whether its mice land **between** those neighbours (`ordered`). In Lung every model put the 21-month mice below the 18-month ones. **Ordinal** regression uses only the order of the ages, not their spacing, so it is the one model built to resist a batch that shifts one age group.

In [ ]:
CLOCKS = M.clock_models()
crow = []
for tissue in sorted(atlas["tissue"].unique()):
    res = pd.read_csv(ag.paths(tissue).figure("age_auroc.csv"), index_col=0)
    good = list(res.index[res["logreg"] >= 0.8])
    if not good:
        continue
    a = sc.read_h5ad(ag.paths(tissue).processed)
    clean = ag.feature_mask(a, "clean")
    for ct in good:
        X, y, mice = ag.prep(a, ct, ages=ag.ADULT, target="months", features=clean, **ag.main_kw(ct, tissue))
        if len(np.unique(mice)) < 4 or len(np.unique(y)) < 3:
            continue
        adult = np.sort(np.unique(y))
        for name, make in CLOCKS.items():
            pred = np.zeros(len(y))
            for tr, te in LeaveOneGroupOut().split(X, y, mice):
                pred[te] = make().fit(X[tr], y[tr]).predict(X[te])
            pm = pd.DataFrame({"m": mice, "t": y, "p": pred}).groupby("m").agg(t=("t", "first"), p=("p", "median"))
            naive = np.mean([abs(v - pm["t"].drop(m).mean()) for m, v in pm["t"].items()])
            ordered = []
            for i in range(1, len(adult) - 1):
                te = y == adult[i]
                if len(np.unique(y[~te])) >= 2:
                    p = np.median(make().fit(X[~te], y[~te]).predict(X[te]))
                    ordered.append(adult[i - 1] < p < adult[i + 1])
            crow.append({"tissue": tissue, "cell_type": ct, "model": name, "mae": (pm.p - pm.t).abs().mean(),
                         "naive": naive, "spearman": spearmanr(pm.t, pm.p)[0],
                         "held_out_ordered": np.mean(ordered) if ordered else np.nan, "holds": len(ordered)})
    print(tissue, "done")
    del a
clocks = pd.DataFrame(crow)
clocks.round(3).to_csv(A.figure("models_clocks.csv"), index=False)

In [ ]:
base_c = clocks[clocks["model"] == "ridge (05c)"].set_index(["tissue", "cell_type"])
csum = []
for name, d in clocks.groupby("model", sort=False):
    d = d.set_index(["tissue", "cell_type"])
    diff = (d["mae"] - base_c.loc[d.index, "mae"]).dropna()
    csum.append({"model": name, "rows": len(d), "mae": d["mae"].mean(), "beats_naive": (d["mae"] < d["naive"]).mean(),
                 "spearman": d["spearman"].mean(), "held_out_ordered": d["held_out_ordered"].mean(),
                 "lower_mae_than_ridge": int((diff < 0).sum()) if name != "ridge (05c)" else np.nan,
                 "wilcoxon_p": wilcoxon(diff).pvalue if name != "ridge (05c)" and (diff != 0).any() else np.nan})
csum = pd.DataFrame(csum).set_index("model")
csum.round(3).to_csv(A.figure("models_clocks_summary.csv"))
csum.round(3)

### Findings

19 rows matched to Kimmel (classifiers), 53 rows with AUROC ≥ 0.8 in every tissue (clocks; the clock features drop the sex-chromosome genes since the critical review). Settings fixed before running; the elastic net solver was switched from stochastic gradient descent to saga after a smoke test showed it had not converged, before this comparison was run.

- **No classifier beats the linear baseline.** On unseen age groups, L2 logistic regression scores 0.747; elastic net 0.720, LightGBM 0.705, attention MIL 0.661 and the mouse-level model 0.657 are lower in 14–16 of 19 rows (the challengers win in only 3–5). A Wilcoxon test over rows gives p = 0.036 to 0.001, but the 19 rows come from 3 tissues and share mice, so those p-values overstate the evidence; the row counts are the result. In another lab they tie it (elastic net 0.616, LightGBM 0.619, MIL 0.597 vs 0.618; p ≥ 0.40) or lose (mouse-level 0.568, lower in 15 of 19 rows).
- **Read as:** with 10–16 mice per row and age tied to batch, extra flexibility buys nothing that generalizes; the neural network, the most flexible model, is among the worst on unseen age groups. The bottleneck is independent samples, not model capacity.
- **Batch check vs transfer, per model:** only the mouse-level model shows a nominal correlation (ρ 0.47, p = 0.04; others −0.02 to 0.25). That is one of five uncorrected tests, so it is not counted as a finding.
- **Ordinal regression improves the clock.** Using only the order of ages, it lowers the per-mouse error in 42 of 53 rows (5.2 vs 5.9 months) and ranks mice better (ρ 0.67 vs 0.63). Rows share mice, so the tissue is the safer unit: ordinal wins in the majority of rows in 7 of 9 tissues (every row in 4), and not in the single Heart or Liver row. LightGBM ties ridge (5.9, p = 0.41); kernel ridge (6.7) and the MLP (6.8) are worse. (With sex genes: 45 of 52 rows, 5.3 vs 6.0.)
- **No model fixes the held-out-age inversion.** Each places a held-out age group between its neighbours only a little more than half the time (0.48–0.59 across models; 0.49–0.54 with sex genes): the same batch problem as in 05c, barely touched by model choice.